# Task 117: pyfemu_vibration — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Structural vibration modal identification using FEM updating

**Paper**: No dedicated paper — standard finite element model updating from modal data
**Repository**: https://github.com/bjorntsv/pyfemu

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 26.46 dB
- **SSIM**: N/A
- **Evaluation**: Vibration-based damage identification (CC=0.986, avg_MAC=0.9999, damage_detection=100%)

### Mathematical Formulation

**Parameter estimation** fits model parameters $\boldsymbol{\theta}$ to data:

$$\hat{\boldsymbol{\theta}} = \arg\min_{\boldsymbol{\theta}} \sum_{i=1}^{N} \left(\frac{y_i - f(t_i; \boldsymbol{\theta})}{\sigma_i}\right)^2$$

**Fisher information matrix**: $\mathcal{I}_{jk} = \sum_i \frac{1}{\sigma_i^2} \frac{\partial f_i}{\partial\theta_j}\frac{\partial f_i}{\partial\theta_k}$

**Cramer-Rao bound**: $\text{Var}(\hat\theta_j) \geq [\mathcal{I}^{-1}]_{jj}$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
pyfemu_vibration — Vibration-Based Damage Identification
==========================================================
From measured modal parameters (natural frequencies, mode shapes) of a beam,
identify damage locations and severities by updating a 1-D Euler-Bernoulli
beam FEM model.

Physics:
  Forward:
    - N-element Euler-Bernoulli beam  →  global K, M matrices
    - Damage in element i  →  K_i × (1 − d_i),  d_i ∈ [0, 1]
    - Eigenvalue problem:  (K − ω² M) φ = 0

  Inverse:
    - Minimise  Σ (ω_obs − ω_model)² + λ Σ (1 − MAC)
    - L-BFGS-B with bounds  d_i ∈ [0, 0.95]
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`euler_bernoulli_element`, `assemble`, `mac_value`, `objective`, `main`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1.  FEM ASSEMBLY
# ═══════════════════════════════════════════════════════════════════
def euler_bernoulli_element(L_e, EI, rhoA):
    """
    4×4 element stiffness and consistent mass matrices for an
    Euler-Bernoulli beam element of length L_e.
    """
    k_e = (EI / L_e ** 3) * np.array([
        [ 12,    6*L_e,  -12,    6*L_e],
        [  6*L_e, 4*L_e**2, -6*L_e, 2*L_e**2],
        [-12,   -6*L_e,   12,   -6*L_e],
        [  6*L_e, 2*L_e**2, -6*L_e, 4*L_e**2]
    ])

    m_e = (rhoA * L_e / 420.0) * np.array([
        [156,    22*L_e,   54,   -13*L_e],
        [ 22*L_e,  4*L_e**2,  13*L_e, -3*L_e**2],
        [ 54,    13*L_e,  156,   -22*L_e],
        [-13*L_e, -3*L_e**2, -22*L_e,  4*L_e**2]
    ])
    return k_e, m_e

def assemble(n_elem, L_total, EI, rhoA, damage_vec):
    """
    Assemble global K and M for a simply-supported beam.
    damage_vec[i] ∈ [0, 1] reduces stiffness of element i.
    """
    n_dof = 2 * (n_elem + 1)   # 2 DOF per node (w, θ)
    L_e   = L_total / n_elem
    K_global = np.zeros((n_dof, n_dof))
    M_global = np.zeros((n_dof, n_dof))

    for i in range(n_elem):
        EI_eff = EI * (1.0 - damage_vec[i])
        k_e, m_e = euler_bernoulli_element(L_e, EI_eff, rhoA)
        idx = [2*i, 2*i+1, 2*i+2, 2*i+3]
        for a in range(4):
            for b in range(4):
                K_global[idx[a], idx[b]] += k_e[a, b]
                M_global[idx[a], idx[b]] += m_e[a, b]

    # Simply-supported BC: w=0 at node 0 and node N
    bc_dofs = [0, 2 * n_elem]   # translational DOFs at ends
    free = [d for d in range(n_dof) if d not in bc_dofs]
    K_free = K_global[np.ix_(free, free)]
    M_free = M_global[np.ix_(free, free)]
    return K_free, M_free, free

# ═══════════════════════════════════════════════════════════════════
# 2.  MAC (Modal Assurance Criterion)
# ═══════════════════════════════════════════════════════════════════
def mac_value(phi_a, phi_b):
    """MAC between two mode shape vectors."""
    num = np.dot(phi_a, phi_b) ** 2
    den = np.dot(phi_a, phi_a) * np.dot(phi_b, phi_b)
    return num / (den + 1e-30)

# ═══════════════════════════════════════════════════════════════════
# 3.  INVERSE OBJECTIVE
# ═══════════════════════════════════════════════════════════════════
def objective(d_vec, n_elem, L_total, EI, rhoA, n_modes, freqs_obs, modes_obs):
    """
    Misfit = frequency error + mode-shape MAC error.
    """
    freqs_mod, modes_mod = solve_modal(n_elem, L_total, EI, rhoA, d_vec, n_modes)

    # Frequency part
    freq_err = np.sum(((freqs_obs - freqs_mod) / (freqs_obs + 1e-30)) ** 2)

    # MAC part
    mac_err = 0.0
    for j in range(n_modes):
        # Ensure consistent sign
        if np.dot(modes_obs[:, j], modes_mod[:, j]) < 0:
            modes_mod[:, j] *= -1
        mac_err += (1.0 - mac_value(modes_obs[:, j], modes_mod[:, j]))

    return freq_err + 0.5 * mac_err

# ═══════════════════════════════════════════════════════════════════
# 6.  MAIN
# ═══════════════════════════════════════════════════════════════════
def main():
    print("=" * 60)
    print("pyfemu_vibration — Vibration-Based Damage Identification")
    print("=" * 60)

    EI   = E_BEAM * I_SECT
    rhoA = RHO_BEAM * AREA

    # 1. Ground-truth damage vector
    d_gt = np.zeros(N_ELEM)
    for elem, severity in GT_DAMAGE.items():
        d_gt[elem] = severity
    print(f"  GT damage: {GT_DAMAGE}")

    # 2. Undamaged modal data (for reference)
    d_zero = np.zeros(N_ELEM)
    freqs_undamaged, _ = solve_modal(N_ELEM, L_TOTAL, EI, rhoA, d_zero, N_MODES)
    print(f"  Undamaged frequencies: {np.round(freqs_undamaged, 2)} Hz")

    # 3. Forward: damaged modal data
    print("[1/4] Computing damaged modal data ...")
    freqs_gt, modes_gt = solve_modal(N_ELEM, L_TOTAL, EI, rhoA, d_gt, N_MODES)
    print(f"  Damaged frequencies:   {np.round(freqs_gt, 2)} Hz")

    # 4. Add noise to observations
    freq_noise_abs = FREQ_NOISE * freqs_gt * np.random.randn(N_MODES)
    freqs_obs = freqs_gt + freq_noise_abs

    mode_noise = MODE_NOISE * np.random.randn(*modes_gt.shape)
    modes_obs = modes_gt + mode_noise
    for j in range(N_MODES):
        modes_obs[:, j] /= np.max(np.abs(modes_obs[:, j])) + 1e-30

    # 5. Inverse: optimise damage parameters
    print("[2/4] Running L-BFGS-B damage identification ...")
    d0 = np.zeros(N_ELEM)  # start undamaged
    bounds = [(0.0, 0.95)] * N_ELEM
    result = minimize(
        objective, d0,
        args=(N_ELEM, L_TOTAL, EI, rhoA, N_MODES, freqs_obs, modes_obs),
        method="L-BFGS-B", bounds=bounds,
        options={"maxiter": 500, "ftol": 1e-14, "gtol": 1e-10}
    )
    d_recon = result.x
    print(f"  Optimisation converged: {result.success}  (nit={result.nit})")
    print(f"  Identified damage >0.05: ", end="")
    for i in range(N_ELEM):
        if d_recon[i] > 0.05:
            print(f"elem {i}={d_recon[i]:.3f}  ", end="")
    print()

    # 6. Recompute modal data with identified damage
    freqs_recon, modes_recon = solve_modal(N_ELEM, L_TOTAL, EI, rhoA, d_recon, N_MODES)

    # 7. Metrics
    print("[3/4] Computing metrics ...")
    metrics = compute_metrics(d_gt, d_recon, freqs_gt, freqs_recon, modes_gt, modes_recon, N_MODES)
    print(f"  PSNR = {metrics['PSNR']:.2f} dB")
    print(f"  CC   = {metrics['CC']:.4f}")
    print(f"  RMSE = {metrics['RMSE']:.6f}")
    print(f"  Freq RMSE = {metrics['freq_RMSE_Hz']:.4f} Hz")
    print(f"  Avg MAC   = {metrics['avg_MAC']:.6f}")
    print(f"  Detection = {metrics['damage_detection_pct']:.0f}%")

    # 8. Save
    print("[4/4] Saving results ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), d_gt)
        np.save(os.path.join(d, "recon_output.npy"), d_recon)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    visualize(d_gt, d_recon, freqs_gt, freqs_obs, freqs_recon,
              modes_gt, modes_recon, N_MODES, metrics)

    print("Done ✓")
    return metrics

## 4. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `solve_modal`

In [ ]:
def solve_modal(n_elem, L_total, EI, rhoA, damage_vec, n_modes):
    """Solve eigenvalue problem → natural frequencies + mode shapes."""
    K, M, free_dofs = assemble(n_elem, L_total, EI, rhoA, damage_vec)
    eigvals, eigvecs = eigh(K, M)
    # Take first n_modes
    omega2 = eigvals[:n_modes]
    freqs  = np.sqrt(np.abs(omega2)) / (2 * np.pi)
    modes  = eigvecs[:, :n_modes]
    # Normalise mode shapes
    for j in range(n_modes):
        modes[:, j] /= np.max(np.abs(modes[:, j])) + 1e-30
    return freqs, modes

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 4.  METRICS
# ═══════════════════════════════════════════════════════════════════
def compute_metrics(d_gt, d_recon, freqs_gt, freqs_recon, modes_gt, modes_recon, n_modes):
    """Damage vector metrics + frequency + MAC metrics."""
    # PSNR on damage vector
    mse = np.mean((d_gt - d_recon) ** 2)
    data_range = max(np.max(d_gt) - np.min(d_gt), 0.01)
    psnr = 10.0 * np.log10(data_range ** 2 / (mse + 1e-30))

    cc = float(np.corrcoef(d_gt, d_recon)[0, 1]) if np.std(d_gt) > 1e-10 else 0.0
    rmse = float(np.sqrt(mse))

    # Frequency RMSE
    freq_rmse = float(np.sqrt(np.mean((freqs_gt - freqs_recon) ** 2)))

    # Average MAC
    mac_vals = []
    for j in range(n_modes):
        if np.dot(modes_gt[:, j], modes_recon[:, j]) < 0:
            modes_recon[:, j] *= -1
        mac_vals.append(mac_value(modes_gt[:, j], modes_recon[:, j]))
    avg_mac = float(np.mean(mac_vals))

    # Damage localisation accuracy
    gt_damaged   = set(np.where(d_gt > 0.05)[0])
    recon_damaged = set(np.where(d_recon > 0.05)[0])
    if len(gt_damaged) > 0:
        detection_rate = len(gt_damaged & recon_damaged) / len(gt_damaged) * 100
    else:
        detection_rate = 100.0

    return {
        "PSNR": float(psnr),
        "CC": cc,
        "RMSE": rmse,
        "freq_RMSE_Hz": freq_rmse,
        "avg_MAC": avg_mac,
        "damage_detection_pct": detection_rate
    }

# ═══════════════════════════════════════════════════════════════════
# 5.  VISUALISATION
# ═══════════════════════════════════════════════════════════════════
def visualize(d_gt, d_recon, freqs_gt, freqs_obs, freqs_recon,
              modes_gt, modes_recon, n_modes, metrics):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    elem_centers = np.arange(N_ELEM) + 0.5

    # (a) Damage distribution
    ax = axes[0, 0]
    ax.bar(elem_centers - 0.2, d_gt, 0.4, label="True Damage", color="steelblue", alpha=0.8)
    ax.bar(elem_centers + 0.2, d_recon, 0.4, label="Identified Damage", color="salmon", alpha=0.8)
    ax.set_xlabel("Element Index")
    ax.set_ylabel("Damage Parameter d")
    ax.set_title(f"Damage Identification  (PSNR={metrics['PSNR']:.1f} dB, CC={metrics['CC']:.4f})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, N_ELEM)

    # (b) Frequency comparison
    ax = axes[0, 1]
    mode_idx = np.arange(1, n_modes + 1)
    ax.plot(mode_idx, freqs_gt, "bo-", label="GT Frequencies")
    ax.plot(mode_idx, freqs_obs, "g^--", label="Observed (noisy)", alpha=0.7)
    ax.plot(mode_idx, freqs_recon, "rs--", label="Identified Model")
    ax.set_xlabel("Mode Number")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_title(f"Modal Frequencies  (freq RMSE={metrics['freq_RMSE_Hz']:.2f} Hz)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # (c) Mode shapes (first 3)
    ax = axes[1, 0]
    n_dof = modes_gt.shape[0]
    x_nodes = np.linspace(0, L_TOTAL, n_dof)
    colors = ["tab:blue", "tab:orange", "tab:green"]
    for j in range(min(3, n_modes)):
        mg = modes_gt[:, j]
        mr = modes_recon[:, j]
        if np.dot(mg, mr) < 0:
            mr = -mr
        ax.plot(x_nodes, mg, "-", color=colors[j], lw=2,
                label=f"Mode {j+1} GT")
        ax.plot(x_nodes, mr, "--", color=colors[j], lw=2,
                label=f"Mode {j+1} Identified")
    ax.set_xlabel("Position (m)")
    ax.set_ylabel("Mode Shape Amplitude")
    ax.set_title(f"Mode Shapes  (avg MAC={metrics['avg_MAC']:.4f})")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

    # (d) Residual / damage error
    ax = axes[1, 1]
    residual = d_gt - d_recon
    ax.bar(elem_centers, residual, 0.6, color="purple", alpha=0.6)
    ax.axhline(0, color="k", ls="--", lw=0.5)
    ax.set_xlabel("Element Index")
    ax.set_ylabel("Damage Error (GT − Identified)")
    ax.set_title(f"Damage Residual  (RMSE={metrics['RMSE']:.4f}, Detection={metrics['damage_detection_pct']:.0f}%)")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, N_ELEM)

    plt.suptitle("Vibration-Based Damage Identification — FE Model Updating", fontsize=14, y=1.01)
    plt.tight_layout()
    for p in [os.path.join(RESULTS_DIR, "reconstruction_result.png"),
              os.path.join(ASSETS_DIR,  "reconstruction_result.png"),
              os.path.join(ASSETS_DIR,  "vis_result.png")]:
        plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("pyfemu_vibration — Vibration-Based Damage Identification")
print("=" * 60)

EI   = E_BEAM * I_SECT
rhoA = RHO_BEAM * AREA

### 1. Ground-truth damage vector

Intermediate processing step in the pipeline.

In [ ]:
# 1. Ground-truth damage vector
d_gt = np.zeros(N_ELEM)
for elem, severity in GT_DAMAGE.items():
    d_gt[elem] = severity
print(f"  GT damage: {GT_DAMAGE}")

### 2. Undamaged modal data (for reference)

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 2. Undamaged modal data (for reference)
d_zero = np.zeros(N_ELEM)
freqs_undamaged, _ = solve_modal(N_ELEM, L_TOTAL, EI, rhoA, d_zero, N_MODES)
print(f"  Undamaged frequencies: {np.round(freqs_undamaged, 2)} Hz")

### 3. Forward: damaged modal data

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 3. Forward: damaged modal data
print("[1/4] Computing damaged modal data ...")
freqs_gt, modes_gt = solve_modal(N_ELEM, L_TOTAL, EI, rhoA, d_gt, N_MODES)
print(f"  Damaged frequencies:   {np.round(freqs_gt, 2)} Hz")

### 4. Add noise to observations

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 4. Add noise to observations
freq_noise_abs = FREQ_NOISE * freqs_gt * np.random.randn(N_MODES)
freqs_obs = freqs_gt + freq_noise_abs

mode_noise = MODE_NOISE * np.random.randn(*modes_gt.shape)
modes_obs = modes_gt + mode_noise
for j in range(N_MODES):
    modes_obs[:, j] /= np.max(np.abs(modes_obs[:, j])) + 1e-30

### 5. Inverse: optimise damage parameters

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 5. Inverse: optimise damage parameters
print("[2/4] Running L-BFGS-B damage identification ...")
d0 = np.zeros(N_ELEM)  # start undamaged
bounds = [(0.0, 0.95)] * N_ELEM
result = minimize(
    objective, d0,
    args=(N_ELEM, L_TOTAL, EI, rhoA, N_MODES, freqs_obs, modes_obs),
    method="L-BFGS-B", bounds=bounds,
    options={"maxiter": 500, "ftol": 1e-14, "gtol": 1e-10}
)
d_recon = result.x
print(f"  Optimisation converged: {result.success}  (nit={result.nit})")
print(f"  Identified damage >0.05: ", end="")
for i in range(N_ELEM):
    if d_recon[i] > 0.05:
        print(f"elem {i}={d_recon[i]:.3f}  ", end="")
print()

### 6. Recompute modal data with identified damage

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 6. Recompute modal data with identified damage
freqs_recon, modes_recon = solve_modal(N_ELEM, L_TOTAL, EI, rhoA, d_recon, N_MODES)

### 7. Metrics

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 7. Metrics
print("[3/4] Computing metrics ...")
metrics = compute_metrics(d_gt, d_recon, freqs_gt, freqs_recon, modes_gt, modes_recon, N_MODES)
print(f"  PSNR = {metrics['PSNR']:.2f} dB")
print(f"  CC   = {metrics['CC']:.4f}")
print(f"  RMSE = {metrics['RMSE']:.6f}")
print(f"  Freq RMSE = {metrics['freq_RMSE_Hz']:.4f} Hz")
print(f"  Avg MAC   = {metrics['avg_MAC']:.6f}")
print(f"  Detection = {metrics['damage_detection_pct']:.0f}%")

### 8. Save

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 8. Save
print("[4/4] Saving results ...")
for d in [RESULTS_DIR, ASSETS_DIR]:
    np.save(os.path.join(d, "gt_output.npy"), d_gt)
    np.save(os.path.join(d, "recon_output.npy"), d_recon)
    with open(os.path.join(d, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

visualize(d_gt, d_recon, freqs_gt, freqs_obs, freqs_recon,
          modes_gt, modes_recon, N_MODES, metrics)

print("Done ✓")
return metrics

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **pyfemu_vibration**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=26.46 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: No dedicated paper — standard finite element model updating from modal data
- Repository: https://github.com/bjorntsv/pyfemu
